# Day 22 Capstone: Energy-Demand Forecasting with LSTM + Attention

This notebook generates a multi-seasonal hourly energy-demand series completely offline, frames a 168-hour to 24-hour forecasting task, compares a seasonal-naive baseline with a direct LSTM and an LSTM with self-attention, evaluates all models in original demand units, and produces a numbers-backed final analysis.

## 0. Imports, seeds, and offline data generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from IPython.display import display, Markdown

RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)

# Generate about two years of hourly demand, fully offline.
N = 24 * 7 * 104
t = np.arange(N)
trend = 0.0015 * t
daily = 8.0 * np.sin(2 * np.pi * t / 24 - 1.0)
weekly = 4.0 * np.sin(2 * np.pi * t / (24 * 7))
weekend = -3.0 * (((t // 24) % 7) >= 5)
noise = np.random.normal(0, 1.2, N)

demand = (50.0 + trend + daily + weekly + weekend + noise).astype("float32")

print(
    "series length:", demand.shape,
    " mean:", round(float(demand.mean()), 2),
    " min:", round(float(demand.min()), 2),
    " max:", round(float(demand.max()), 2),
)
assert demand.shape == (17472,)

In [ ]:
# Inspect the first four weeks.
plt.figure(figsize=(12, 4))
plt.plot(t[:24 * 28], demand[:24 * 28], linewidth=1)
plt.xlabel("hour")
plt.ylabel("demand")
plt.title("Generated hourly energy demand — first 4 weeks")
plt.tight_layout()
plt.show()

## 1. Frame the supervised problem: 168-hour look-back → 24-hour horizon

In [ ]:
LOOKBACK = 168
HORIZON = 24

def make_windows(series, lookback, horizon):
    X, y = [], []
    for i in range(len(series) - lookback - horizon + 1):
        X.append(series[i:i + lookback])
        y.append(series[i + lookback:i + lookback + horizon])
    return np.array(X, dtype="float32"), np.array(y, dtype="float32")

X_all, y_all = make_windows(demand, LOOKBACK, HORIZON)
print("X_all:", X_all.shape, " y_all:", y_all.shape)
assert X_all.shape == (17281, 168)
assert y_all.shape == (17281, 24)

## 2. Chronological split and train-only normalization

In [ ]:
n = len(X_all)
i_tr = int(n * 0.70)
i_va = int(n * 0.85)

X_tr_raw, y_tr_raw = X_all[:i_tr], y_all[:i_tr]
X_va_raw, y_va_raw = X_all[i_tr:i_va], y_all[i_tr:i_va]
X_te_raw, y_te_raw = X_all[i_va:], y_all[i_va:]

# Fit normalization statistics only on training inputs.
mu = X_tr_raw.mean()
sigma = X_tr_raw.std()

def norm(a):
    return (a - mu) / sigma

def denorm(a):
    return a * sigma + mu

X_tr, y_tr = norm(X_tr_raw), norm(y_tr_raw)
X_va, y_va = norm(X_va_raw), norm(y_va_raw)
X_te, y_te = norm(X_te_raw), norm(y_te_raw)

# Keras recurrent input shape: (samples, timesteps, features)
X_tr = X_tr[..., None]
X_va = X_va[..., None]
X_te = X_te[..., None]

print("train:", X_tr.shape, " val:", X_va.shape, " test:", X_te.shape)
print("norm stats mu:", round(float(mu), 3), " sigma:", round(float(sigma), 3))
print("split indices:", i_tr, i_va, n)

assert np.isclose(X_tr.mean(), 0.0, atol=1e-5)
assert np.isclose(X_tr.std(), 1.0, atol=1e-5)

The split is chronological and `shuffle=False` will also be used during fitting. Normalization uses only `X_tr_raw`, preventing future validation or test statistics from leaking into training.

## 3. Seasonal-naive baseline

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(a - b)))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def mape(a, b):
    return float(np.mean(np.abs((a - b) / a)) * 100)

# Predict the next 24 hours using the final 24 hours of the look-back window.
baseline_pred = X_te_raw[:, -HORIZON:]
y_te_true = y_te_raw

b_mae = mae(y_te_true, baseline_pred)
b_rmse = rmse(y_te_true, baseline_pred)
b_mape = mape(y_te_true, baseline_pred)

print(f"SEASONAL-NAIVE MAE={b_mae:.3f} RMSE={b_rmse:.3f} MAPE={b_mape:.2f}%")

## 4. Direct multi-output LSTM forecaster

In [ ]:
def build_lstm():
    inp = keras.Input(shape=(LOOKBACK, 1))
    x = layers.LSTM(48)(inp)
    x = layers.Dropout(0.1)(x)
    out = layers.Dense(HORIZON)(x)
    model = keras.Model(inp, out, name="lstm_direct")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"],
    )
    return model

lstm = build_lstm()
lstm.summary()

es = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

hist = lstm.fit(
    X_tr,
    y_tr,
    validation_data=(X_va, y_va),
    epochs=15,
    batch_size=128,
    callbacks=[es],
    shuffle=False,
    verbose=2,
)

print(
    "epochs run:", len(hist.history["loss"]),
    " best val_mae(norm):", round(min(hist.history["val_mae"]), 4),
)

## 5. LSTM with self-attention over encoder states

In [ ]:
def build_lstm_attention():
    inp = keras.Input(shape=(LOOKBACK, 1))
    enc = layers.LSTM(48, return_sequences=True)(inp)
    ctx = layers.Attention(name="self_attention")([enc, enc])
    ctx = layers.LayerNormalization(name="attention_norm")(ctx + enc)
    pooled = layers.GlobalAveragePooling1D()(ctx)
    x = layers.Dropout(0.1)(pooled)
    out = layers.Dense(HORIZON)(x)
    model = keras.Model(inp, out, name="lstm_attention")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"],
    )
    return model

lstm_attn = build_lstm_attention()
lstm_attn.summary()

es2 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

hist2 = lstm_attn.fit(
    X_tr,
    y_tr,
    validation_data=(X_va, y_va),
    epochs=15,
    batch_size=128,
    callbacks=[es2],
    shuffle=False,
    verbose=2,
)

print(
    "epochs run:", len(hist2.history["loss"]),
    " best val_mae(norm):", round(min(hist2.history["val_mae"]), 4),
)
print(lstm_attn.count_params(), "params vs", lstm.count_params(), "for plain LSTM")

In [ ]:
def plot_training(history, title):
    epochs = np.arange(1, len(history.history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].plot(epochs, history.history["loss"], label="train")
    axes[0].plot(epochs, history.history["val_loss"], label="validation")
    axes[0].set_title("MSE loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(epochs, history.history["mae"], label="train")
    axes[1].plot(epochs, history.history["val_mae"], label="validation")
    axes[1].set_title("MAE in normalized units")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_training(hist, "Plain LSTM training")
plot_training(hist2, "LSTM + attention training")

## 6. Evaluate all models in original demand units

In [ ]:
def score_model(model, name):
    pred = denorm(model.predict(X_te, verbose=0))
    return name, mae(y_te_true, pred), rmse(y_te_true, pred), mape(y_te_true, pred), pred

lstm_name, l_mae, l_rmse, l_mape, lstm_pred = score_model(lstm, "lstm")
a_name, a_mae, a_rmse, a_mape, attn_pred = score_model(lstm_attn, "lstm+attention")

rows = [
    ("seasonal-naive", b_mae, b_rmse, b_mape),
    (lstm_name, l_mae, l_rmse, l_mape),
    (a_name, a_mae, a_rmse, a_mape),
]

print(f"{'model':16s} {'MAE':>8s} {'RMSE':>8s} {'MAPE%':>8s}")
for nm, a, r, p in rows:
    print(f"{nm:16s} {a:8.3f} {r:8.3f} {p:8.2f}")

best = min(rows, key=lambda z: z[1])
print()
print("lowest MAE:", best[0])

## 7. Forecast plot and inference-regime discussion

In [ ]:
k = 0
truth = y_te_true[k]
naive = baseline_pred[k]
net = attn_pred[k]

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, HORIZON + 1), truth, "o-", label="truth")
plt.plot(np.arange(1, HORIZON + 1), naive, "s--", label="seasonal-naive")
plt.plot(np.arange(1, HORIZON + 1), net, "^--", label="lstm+attention")
plt.xlabel("hours ahead (1..24)")
plt.ylabel("demand")
plt.title("24-hour-ahead forecast — one test window")
plt.legend()
plt.tight_layout()
plt.show()

print("truth[:6] :", np.round(truth[:6], 2))
print("naive[:6] :", np.round(naive[:6], 2))
print("net  [:6] :", np.round(net[:6], 2))

sample_naive_mae = mae(truth, naive)
sample_net_mae = mae(truth, net)
print(f"sample naive MAE={sample_naive_mae:.3f}; sample attention MAE={sample_net_mae:.3f}")

### Plot interpretation and teacher forcing

The plot compares the true 24-hour trajectory with the previous-day seasonal pattern and the attention model's direct forecast. Use the printed first six values and sample MAEs above to state which curve follows the level and daily rise-and-fall more closely; also note any hour where either forecast visibly misses a peak or trough.

Both neural networks here are **direct multi-output forecasters**: each network emits all 24 future values in one forward pass, so teacher forcing is not used during training or inference. In a sequence-to-sequence decoder trained with teacher forcing, the decoder receives the true previous target during training but must consume its own previous prediction during autoregressive inference. That mismatch is exposure bias, and an early prediction error can compound through later forecast steps.

## 8. Analysis

In [ ]:
print("BASELINE MAE=%.3f RMSE=%.3f MAPE=%.2f%%" % (b_mae, b_rmse, b_mape))
for nm, a, r, p in rows[1:]:
    print(
        "%-14s MAE=%.3f RMSE=%.3f MAPE=%.2f%% "
        "(dMAE=%+.3f dRMSE=%+.3f dMAPE=%+.2f pp)"
        % (nm, a, r, p, a - b_mae, r - b_rmse, p - b_mape)
    )

In [ ]:
# Automatically creates a 150–250 word, numbers-backed reflection after training.
plain_delta = l_mae - b_mae
attn_delta = a_mae - b_mae
attn_vs_plain = a_mae - l_mae
winner = min(rows, key=lambda z: z[1])[0]

reflection = f"""
### Analysis

The seasonal-naive baseline achieved MAE {b_mae:.3f}, RMSE {b_rmse:.3f}, and MAPE {b_mape:.2f}%. The plain LSTM produced MAE {l_mae:.3f}, RMSE {l_rmse:.3f}, and MAPE {l_mape:.2f}%, changing the baseline errors by {plain_delta:+.3f} MAE, {l_rmse-b_rmse:+.3f} RMSE, and {l_mape-b_mape:+.2f} percentage points MAPE. The attention model produced MAE {a_mae:.3f}, RMSE {a_rmse:.3f}, and MAPE {a_mape:.2f}%, with corresponding baseline changes of {attn_delta:+.3f}, {a_rmse-b_rmse:+.3f}, and {a_mape-b_mape:+.2f}. Therefore, the lowest-MAE method was {winner}.

Attention changed MAE by {attn_vs_plain:+.3f} relative to the plain LSTM while adding {lstm_attn.count_params()-lstm.count_params():,} parameters. A very small difference should be treated as run-to-run noise rather than strong evidence that attention earned its complexity. Attention would be more valuable for longer horizons, irregular seasonal relationships, or exogenous signals such as temperature, holidays, and tariffs. Evaluation remained honest: windows were split chronologically, normalization statistics came only from training inputs, fitting used no shuffling, and the held-out test set was used only for final scoring. Both networks forecast all 24 steps directly, so they avoid teacher forcing and autoregressive exposure bias; a decoder feeding back its own forecasts could accumulate errors. Next, I would add hour-of-day and day-of-week features and use quantile loss to produce calibrated prediction intervals.
""".strip()

word_count = len(reflection.replace("### Analysis", "").split())
display(Markdown(reflection))
print("Reflection word count:", word_count)
assert 150 <= word_count <= 250, f"Reflection must be 150–250 words; got {word_count}."



## 9. Save models and final checks

In [ ]:
lstm.save("day22_lstm.keras")
lstm_attn.save("day22_lstm_attention.keras")
print("Saved day22_lstm.keras and day22_lstm_attention.keras")
print("Before submitting: run all cells, verify every plot/output, and save this notebook as day22_capstone.ipynb.")